# Multilingual News Intelligence Platform — Corpus Embedding (BGE-M3)

This notebook embeds **54,102 multilingual news chunks** across English, Hindi, Bengali, and Tamil into 1024-dimensional dense vectors using **BGE-M3 (fp16)** on a Kaggle GPU (T4).

### Instructions:
1. Click **Settings** (right panel) -> **Accelerator** -> Select **GPU T4** (or T4 x2).
2. Click **Add Input** (top right) -> Upload `data/chunks.parquet` from this project as a Kaggle Dataset (e.g. named `news-intel-chunks`).
3. Run all cells.
4. Download `vectors.npy` and `chunks_final.parquet` from the **Output** section.
5. Place both downloaded files in the `data/` folder of your project.

In [ ]:
!pip install -q FlagEmbedding transformers torch pandas pyarrow tqdm scikit-learn

In [ ]:
import os
import glob
import torch
import numpy as np
import pandas as pd
from FlagEmbedding import BGEM3FlagModel

print("PyTorch version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))
else:
    print("WARNING: No GPU detected! Please enable GPU accelerator in notebook settings.")

In [ ]:
# Locate chunks.parquet automatically in /kaggle/input or current directory
candidates = (
    glob.glob("/kaggle/input/**/chunks.parquet", recursive=True)
    + glob.glob("chunks.parquet")
    + glob.glob("data/chunks.parquet")
)

if not candidates:
    raise FileNotFoundError(
        "Could not find chunks.parquet! Please click 'Add Input' on the right panel and upload chunks.parquet."
    )

chunks_file = candidates[0]
print(f"Found chunks file at: {chunks_file}")

df = pd.read_parquet(chunks_file)
print(f"Loaded {len(df):,} total chunks.")
print("\nLanguage distribution:")
print(df["lang"].value_counts())

texts = df["text"].fillna("").astype(str).tolist()

In [ ]:
print("Loading BGE-M3 model in fp16...")
model = BGEM3FlagModel(
    "BAAI/bge-m3",
    use_fp16=torch.cuda.is_available(),
    device="cuda" if torch.cuda.is_available() else "cpu",
)

print(f"Embedding {len(texts):,} passages with batch_size=64...")
# Batch size 64 fits comfortably in Kaggle T4 16GB VRAM
output = model.encode(
    texts,
    batch_size=64,
    max_length=512,
    return_dense=True,
    return_sparse=False,
    return_colbert_vecs=False,
)

dense_vectors = np.array(output["dense_vecs"], dtype=np.float32)
print(f"Embedding complete! Shape: {dense_vectors.shape}")

In [ ]:
# Save vectors and chunks to output directory
out_dir = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
vectors_path = os.path.join(out_dir, "vectors.npy")
chunks_path = os.path.join(out_dir, "chunks_final.parquet")

np.save(vectors_path, dense_vectors)
df.to_parquet(chunks_path, index=False)

print("=== FINISHED SUCCESSFULLY ===")
print(f"Saved {len(dense_vectors):,} vectors to: {vectors_path} ({os.path.getsize(vectors_path)/(1024*1024):.2f} MB)")
print(f"Saved {len(df):,} chunk records to: {chunks_path} ({os.path.getsize(chunks_path)/(1024*1024):.2f} MB)")
print("\nNow download both files from the Kaggle Output tab and place them into your project's data/ folder.")